# 华润江中 600750.SH — 技术指标计算与可视化

**指标列表：**
- **RSI (14)** — 相对强弱指标，Wilder 平滑法
- **MACD (12, 26, 9)** — 指数平滑异同移动平均线
- **Bollinger Bands (20, 2)** — 布林带

**数据范围：** 2025-06-30 ~ 2026-06-26，共 241 个交易日

## 1. 环境准备

导入所需库并设置绘图风格。（如未安装 plotly，先执行 `!pip install plotly`）

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# A股颜色约定：红涨绿跌
COLOR_UP   = '#ef5350'
COLOR_DOWN = '#26a69a'

## 2. 加载股价数据

从本地 CSV 文件读取日线数据，并按交易日期排序。

In [ ]:
# 加载已存储的股价数据
csv_path = "600750_daily.csv"
df = pd.read_csv(csv_path)

# 日期转换与排序
df['trade_date'] = pd.to_datetime(df['trade_date'], format='%Y%m%d')
df = df.sort_values('trade_date').reset_index(drop=True)

# 提取核心数组（便于后续向量化计算）
close = df['close'].values
high  = df['high'].values
low   = df['low'].values
dates = df['trade_date']

print(f"数据加载完成：{len(df)} 个交易日")
print(f"日期范围：{dates.min().strftime('%Y-%m-%d')} ~ {dates.max().strftime('%Y-%m-%d')}")
print(f"价格范围：¥{close.min():.2f} ~ ¥{close.max():.2f}")
df.head()

## 3. 计算 RSI (Relative Strength Index)

**RSI 公式（Wilder 平滑法）：**

1. 计算每日涨跌：$\Delta_t = close_t - close_{t-1}$
2. 分离涨跌幅：$gain_t = \max(\Delta_t, 0)$，$loss_t = \max(-\Delta_t, 0)$
3. 初始平均（第 14 天）：$AvgGain = rac{1}{14}\sum_{i=1}^{14} gain_i$
4. Wilder 平滑递推：
   $$AvgGain_t = rac{AvgGain_{t-1} 	imes 13 + gain_t}{14}$$

5. 计算 RSI：
   $$RS = rac{AvgGain}{AvgLoss},\quad RSI = 100 - rac{100}{1+RS}$$

**信号解读：** RSI > 70 超买 | RSI < 30 超卖 | 50 为多空分界

In [ ]:
def compute_rsi(prices: np.ndarray, period: int = 14) -> np.ndarray:
    """Wilder's smoothing RSI 计算"""
    n = len(prices)
    delta = np.diff(prices, prepend=prices[0])
    gain = np.where(delta > 0, delta, 0)
    loss = np.where(delta < 0, -delta, 0)

    avg_gain = np.full(n, np.nan)
    avg_loss = np.full(n, np.nan)

    # 初始 SMA
    avg_gain[period] = gain[1:period + 1].mean()
    avg_loss[period] = loss[1:period + 1].mean()

    # Wilder 平滑
    for i in range(period + 1, n):
        avg_gain[i] = (avg_gain[i - 1] * (period - 1) + gain[i]) / period
        avg_loss[i] = (avg_loss[i - 1] * (period - 1) + loss[i]) / period

    rs = avg_gain / avg_loss
    rsi = 100.0 - 100.0 / (1.0 + rs)
    return rsi

df['rsi'] = compute_rsi(close, 14)

# 查看最近 5 天 RSI
df[['trade_date', 'close', 'rsi']].tail()

## 4. 计算 MACD

**MACD 公式（EMA 法）：**

1. **DIF**（快线）= $EMA_{12} - EMA_{26}$
2. **DEA**（慢线/信号线）= $EMA_9(DIF)$
3. **MACD 柱**（Histogram）= $DIF - DEA$

**信号解读：**
- DIF 上穿 DEA → **金叉**（买入信号）
- DIF 下穿 DEA → **死叉**（卖出信号）
- 柱体由负转正 → 动能转多；由正转负 → 动能转空

In [ ]:
def compute_macd(prices: np.ndarray, fast: int = 12, 
                  slow: int = 26, signal: int = 9):
    """EMA-based MACD 计算"""
    s = pd.Series(prices)
    ema_fast = s.ewm(span=fast, adjust=False).mean()
    ema_slow = s.ewm(span=slow, adjust=False).mean()
    dif = ema_fast - ema_slow
    dea = dif.ewm(span=signal, adjust=False).mean()
    hist = dif - dea
    return dif.values, dea.values, hist.values

df['macd'], df['macd_signal'], df['macd_hist'] = compute_macd(close)

df[['trade_date', 'close', 'macd', 'macd_signal', 'macd_hist']].tail()

## 5. 计算布林带 (Bollinger Bands)

**布林带公式：**

1. **中轨** = $MA_{20}$ （20 日简单移动平均）
2. **上轨** = 中轨 + $2 	imes \sigma_{20}$
3. **下轨** = 中轨 - $2 	imes \sigma_{20}$
4. **%B** = $\dfrac{Close - 下轨}{上轨 - 下轨}$ （股价在带内的相对位置）
5. **带宽** = $\dfrac{上轨 - 下轨}{中轨} 	imes 100\%$

**信号解读：**
- %B > 1 → 突破上轨（超买/强势突破）
- %B < 0 → 跌破下轨（超卖/弱势破位）
- 带宽收窄 → 变盘前兆；带宽扩张 → 趋势加速

In [ ]:
def compute_bollinger(prices: np.ndarray, period: int = 20,
                        num_std: float = 2.0):
    """布林带计算"""
    s = pd.Series(prices)
    middle = s.rolling(window=period).mean()
    std = s.rolling(window=period).std()
    upper = middle + num_std * std
    lower = middle - num_std * std
    bandwidth = (upper - lower) / middle * 100
    percent_b = (s - lower) / (upper - lower)
    return middle.values, upper.values, lower.values,            bandwidth.values, percent_b.values

df['bb_mid'], df['bb_up'], df['bb_lo'],     df['bb_width'], df['bb_pct_b'] = compute_bollinger(close)

df[['trade_date', 'close', 'bb_up', 'bb_mid', 
    'bb_lo', 'bb_width', 'bb_pct_b']].tail()

## 6. 技术指标可视化

四面板交互式图表：
- **面板 1**：K线 + 布林带（上/中/下轨）
- **面板 2**：RSI(14) + 超买超卖参考线
- **面板 3**：MACD 柱 + DIF + DEA
- **面板 4**：成交量（红涨绿跌）

In [ ]:
# ── 创建四面板图表 ──
fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    row_heights=[0.4, 0.2, 0.2, 0.2],
    subplot_titles=(
        "K线 & 布林带 (Bollinger Bands)",
        "RSI (14)",
        "MACD (12, 26, 9)",
        "成交量 (Vol)"
    )
)

# ── 面板 1: K线 + 布林带 ──
fig.add_trace(go.Candlestick(
    x=dates, open=df['open'], high=df['high'],
    low=df['low'], close=df['close'],
    name="K线",
    increasing_line_color=COLOR_UP,
    decreasing_line_color=COLOR_DOWN,
    showlegend=True
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=dates, y=df['bb_up'], mode='lines',
    line=dict(color='gray', width=1, dash='dash'),
    name='上轨'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=dates, y=df['bb_mid'], mode='lines',
    line=dict(color='#4285f4', width=1),
    name='中轨 MA20'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=dates, y=df['bb_lo'], mode='lines',
    line=dict(color='gray', width=1, dash='dash'),
    name='下轨',
    fill='tonexty',
    fillcolor='rgba(66,133,244,0.05)'
), row=1, col=1)

# ── 面板 2: RSI ──
fig.add_trace(go.Scatter(
    x=dates, y=df['rsi'], mode='lines',
    line=dict(color='#7c4dff', width=1.5),
    name='RSI(14)'
), row=2, col=1)
fig.add_hline(y=70, line_dash="dot", line_color="rgba(239,83,80,0.5)",
              row=2, col=1, annotation_text="超买 70")
fig.add_hline(y=30, line_dash="dot", line_color="rgba(38,166,154,0.5)",
              row=2, col=1, annotation_text="超卖 30")
fig.add_hline(y=50, line_dash="dot", line_color="rgba(0,0,0,0.12)",
              row=2, col=1)

# ── 面板 3: MACD ──
hist_colors = [COLOR_UP if v >= 0 else COLOR_DOWN for v in df['macd_hist']]
fig.add_trace(go.Bar(
    x=dates, y=df['macd_hist'], name='MACD 柱',
    marker_color=hist_colors
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=dates, y=df['macd'], mode='lines',
    line=dict(color='#2196f3', width=1.5),
    name='DIF'
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=dates, y=df['macd_signal'], mode='lines',
    line=dict(color='#ff9800', width=1.5),
    name='DEA'
), row=3, col=1)
fig.add_hline(y=0, line_color="rgba(0,0,0,0.15)", row=3, col=1)

# ── 面板 4: 成交量 ──
vol_colors = [COLOR_UP if df.loc[i, 'close'] >= df.loc[i, 'open']
              else COLOR_DOWN for i in range(len(df))]
fig.add_trace(go.Bar(
    x=dates, y=df['vol'], name='成交量(手)',
    marker_color=vol_colors
), row=4, col=1)

# ── 全局布局 ──
fig.update_layout(
    title=dict(
        text="<b>华润江中 600750.SH — 技术指标全景</b><br>"
             "<sup>RSI(14) · MACD(12,26,9) · Bollinger Bands(20,2)</sup>",
        font=dict(size=18)
    ),
    xaxis_rangeslider_visible=False,
    height=1000,
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='center', x=0.5),
    template='plotly_white',
)

fig.update_xaxes(
    rangeselector=dict(buttons=list([
        dict(count=1, label="1月", step="month", stepmode="backward"),
        dict(count=3, label="3月", step="month", stepmode="backward"),
        dict(count=6, label="6月", step="month", stepmode="backward"),
        dict(count=1, label="1年", step="year", stepmode="backward"),
        dict(step="all", label="全部"),
    ]))
)

fig.update_yaxes(title_text="价格 (¥)", row=1, col=1)
fig.update_yaxes(title_text="RSI", row=2, col=1, range=[0, 100])
fig.update_yaxes(title_text="MACD", row=3, col=1)
fig.update_yaxes(title_text="成交量(手)", row=4, col=1)

fig.show()

## 7. 最新交易日指标摘要

In [ ]:
latest = df.iloc[-1]
print("=" * 50)
print(f"  日期:     {latest['trade_date'].strftime('%Y-%m-%d')}")
print(f"  收盘价:   ¥{latest['close']:.2f}")
print(f"  RSI(14):  {latest['rsi']:.2f}")
print(f"  MACD DIF: {latest['macd']:.4f}")
print(f"  MACD DEA: {latest['macd_signal']:.4f}")
print(f"  MACD 柱:  {latest['macd_hist']:.4f}")
print(f"  布林上轨: ¥{latest['bb_up']:.2f}")
print(f"  布林中轨: ¥{latest['bb_mid']:.2f}")
print(f"  布林下轨: ¥{latest['bb_lo']:.2f}")
print(f"  带宽:     {latest['bb_width']:.2f}%")
print(f"  %B:       {latest['bb_pct_b']:.4f}")
print("=" * 50)

# 信号解读
print("\n📊 信号解读:")
rsi = latest['rsi']
if rsi > 70:
    print(f"  RSI = {rsi:.1f}  → 超买区域")
elif rsi < 30:
    print(f"  RSI = {rsi:.1f}  → 超卖区域")
else:
    print(f"  RSI = {rsi:.1f}  → 中性区间")

macd, sig, hist = latest['macd'], latest['macd_signal'], latest['macd_hist']
if hist > 0 and macd > sig:
    print(f"  MACD 柱正 + DIF>DEA → 多头信号")
elif hist < 0 and macd < sig:
    print(f"  MACD 柱负 + DIF<DEA → 空头信号")

pct_b = latest['bb_pct_b']
if pct_b > 1:
    print(f"  %B > 1 → 突破上轨，超买/强势")
elif pct_b < 0:
    print(f"  %B < 0 → 跌破下轨，超卖/弱势")
else:
    print(f"  %B = {pct_b:.2f} → 带内运行")